# Evaluating the Robustness of Isolation Forest in UEBA
## A Case Study on the NSL-KDD Dataset

**Author:** Judah Idowu · [judahidowu.lovable.app](https://judahidowu.lovable.app)

---

This notebook walks through the companion code for the manuscript:

1. NSL-KDD data pipeline (preprocessing + synthetic replication)
2. Baseline Isolation Forest UEBA — reproducing the 92.51% accuracy finding
3. Feature sensitivity matrix — the 10 highest-leverage inputs
4. Security Decay findings — three evasion scenarios and their non-linear profile
5. The Extremity Paradox and the Goldilocks Zone
6. Future research vectors

In [ ]:
import sys
sys.path.append('src')

import json
import numpy as np
import pandas as pd
from pipeline import NSLKDDPipeline
from detector import UEBADetector, FeatureSensitivityAnalyzer, extract_true_positives
from experiment import MANUSCRIPT_BASELINE, MANUSCRIPT_SENSITIVITY, MANUSCRIPT_EVASION

# Load data (synthetic replication if real NSL-KDD not present)
pipeline = NSLKDDPipeline()
X_train, X_test, y_test, feature_names = pipeline.load()

print(f"\nTest set: {len(y_test):,} samples")
print(f"  Normal : {(y_test==0).sum():,}")
print(f"  Attack : {(y_test==1).sum():,}")

## Baseline Evaluation

The model is trained exclusively on normal traffic — the UEBA philosophy. No attack signatures are seen during training.

In [ ]:
det = UEBADetector(n_estimators=100, contamination=0.01)
det.fit(X_train)
metrics = det.evaluate(X_test, y_test)

print("\nComputed vs. Manuscript:")
print(f"{'Metric':<25} {'Computed':>10} {'Manuscript':>12}")
print("-" * 50)
for key in ['accuracy', 'precision_normal', 'recall_normal', 'f1_normal',
            'precision_attack', 'recall_attack', 'f1_attack']:
    comp = metrics.get(key, 0)
    ms   = MANUSCRIPT_BASELINE.get(key, 0)
    print(f"{key:<25} {comp:>10.4f} {ms:>12.4f}")

print(f"\nConfusion matrix:")
print(f"  TN={metrics['true_negatives']:,}  FP={metrics['false_positives']:,}")
print(f"  FN={metrics['false_negatives']:,}   TP={metrics['true_positives']:,}")
print(f"\nNote: Synthetic data will show lower accuracy than the manuscript.")
print(f"Place real NSL-KDD files in data/ to reproduce 92.51%.")

In [ ]:
from IPython.display import Image
Image('figures/confusion_matrix.png')

In [ ]:
Image('figures/classification_report.png')

## Feature Sensitivity Matrix

Pearson correlation between each feature and the raw Isolation Forest anomaly score. Higher |r| = higher leverage for evasion.

In [ ]:
print("Manuscript Top 10 (real NSL-KDD):")
for rank, (feat, r) in enumerate(MANUSCRIPT_SENSITIVITY.items(), 1):
    print(f"  {rank:<4} {feat:<40} |r| = {r:.4f}")

In [ ]:
Image('figures/feature_sensitivity.png')

## Security Decay: Three Evasion Scenarios

The adversarial phase targets the 49,870 True Positive (correctly detected) attack samples and measures how many evade detection after feature perturbation.

In [ ]:
print("Manuscript evasion findings:\n")
for exp_id, exp in MANUSCRIPT_EVASION.items():
    print(f"  {exp_id.upper()}:")
    print(f"    Method   : {exp['label']}")
    print(f"    Evaded   : {exp['evaded']:,} / {exp['total_tp']:,}")
    print(f"    Rate     : {exp['evasion_rate']*100:.2f}%")
    if 'insight' in exp:
        print(f"    Insight  : {exp['insight']}")
    print()

In [ ]:
Image('figures/evasion_comparison.png')

In [ ]:
Image('figures/security_decay.png')

**The non-linear vulnerability profile:**

- Experiment A (single-feature) → highest evasion (0.83%), linear growth
- Experiment B (multi-feature) → lower evasion (0.61%), non-linear — coordinated changes produce conflicting feature combinations detected as a novel anomaly
- Experiment C (extreme stealth) → lowest evasion (0.11%) — the **Extremity Paradox**

The Extremity Paradox is the central finding: Isolation Forest isolates outliers via short partition paths. Near-zero values across multiple features create structural outliers just as extreme high values do. More stripping = more detectable.

## The Goldilocks Zone

In [ ]:
Image('figures/goldilocks_zone.png')

Maximum evasion occurs in the **16–28% reduction range** across the five high-leverage features. Below this window, perturbation is insufficient to cross the decision boundary. Above it, over-perturbation triggers the Extremity Trap.

This narrow window has a direct defensive implication: monitoring for attack traffic in precisely this perturbation range is more targeted than blanket anomaly detection.

## Contamination Hyperparameter Sensitivity

In [ ]:
Image('figures/contamination_sweep.png')

The `contamination=0.01` setting prioritises precision (674 FP) over recall (8,760 FN) — appropriate for a production UEBA context where false alarms carry operational cost. Increasing contamination to 0.10 roughly halves false negatives at the cost of 10× more false positives.

---

## Future Research Vectors

Three specific directions for extension:

1. **Goldilocks Boundary Mapping** — grid search across feature subsets to define the full evasion surface
2. **Black-Box Transferability** — test whether these white-box evasion samples transfer to OC-SVM, LOF, HBOS
3. **Adversarial Training** — inject evasion samples into training data and retest baseline accuracy

---
*Companion code for: Idowu, J. (manuscript in preparation). "Evaluating the Robustness of Isolation Forest in UEBA: A Case Study on the NSL-KDD Dataset."*